### Manual backprop practice

here we will be replacing loss.backward() with our own calcs

In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [3]:
# Start with our previous MLP
words = open('names.txt', 'r').read().splitlines()
chars = sorted(list(set(''.join(words))))
s_to_i = {s: i+1 for i, s in enumerate(chars)}
s_to_i['.'] = 0
i_to_s = {i:s for s, i in s_to_i.items()}

In [4]:
block_size = 3

# create the dataset, now we have 228146 datapoints
# implement test and training splits
def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        for ch in w+'.':
            ix = s_to_i[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

# create the datasets
X_train, Y_train = build_dataset(words[:n1])
X_val, Y_val = build_dataset(words[n1:n2])
X_test, Y_test = build_dataset(words[n2:])

In [5]:
# util func to compare gradients to torch version
def cmp(s, dt, t):
    exact = torch.all(dt == t.grad).item()
    approximate = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(exact):5s} | approximate {str(approximate):5s} | maxdiff: {maxdiff}')

In [6]:
# same code as last time - 
num_chars = 27
emb_dim = 10
hidden_layer_size = 64

# initialise
g = torch.Generator().manual_seed(2147483647)

# embedding matrix
C = torch.randn((num_chars, emb_dim), generator=g)

# layer 1
W1 = torch.randn((emb_dim * block_size, hidden_layer_size), generator=g) / (5/3)/(emb_dim*num_chars)**0.5
b1 = torch.randn(hidden_layer_size, generator=g) * 0.1 
# layer 2
W2 = torch.randn((hidden_layer_size, num_chars), generator=g) * 0.1
b2 = torch.randn(num_chars, generator=g) * 0.1 # 

# batch norm params - init as ones and zeros so our preactivations start off standard normal
bngain = torch.randn((1, hidden_layer_size)) * 0.1 + 1.0
bnbias = torch.randn((1, hidden_layer_size)) * 0.1

# note we are setting the params to random rather than ones or zeros - this is so we can actually inspect the gradients etc. Zeros can mask an incorrect backward pass

# running batch norm totals
# running_mean = torch.zeros(hidden_layer_size)
# running_var = torch.ones(hidden_layer_size)

# full params
params = [C, W1, b1, W2, b2, bngain, bnbias]
for p in params:
    p.requires_grad = True

In [7]:
# set up batches
batch_size = 32
n = batch_size

ix = torch.randint(0, X_train.shape[0], (batch_size, ), generator=g)
Xb, Yb, = X_train[ix], Y_train[ix]

In [8]:
# forward pass, broken up into its smallest pieces to make it easier for backprop

emb = C[Xb]
embcat = emb.view(emb.shape[0], -1)

# linear layer 1
hprebn = embcat @ W1 + b1

# batchnorm ------------------------------
# get mean of batch 
bnmeani = 1/n * hprebn.sum(0, keepdim=True) # instead of using .mean(), we use sum/n elementary operations instead

# get variance
bndiff = hprebn - bnmeani # take x - mu
bndiff2 = bndiff**2  # (x - mu)^2
bnvar = 1/(n-1) * bndiff2.sum(0, keepdim=True) # sum((x-mu)^2)/n-1 for full population var
bnvar_inv = (bnvar + 1e-5)**-0.5 # 1/sqrt(var + eps) to give inverted std dev

bnraw = bndiff * bnvar_inv #  (x - mean)/stddev
hpreact = bngain * bnraw + bnbias # gain * batchnormalised + bias
# ----------------------------------------
# tanh
h = torch.tanh(hpreact)

# layer 2
logits = h @ W2 + b2

# loss - replacing F.cross_entropy -----------------
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # normalise to keep them stable - if we have extreme logits, we can get NaN when we softmax to get probabilites
counts = norm_logits.exp() # take e^logit to get more reasonable count numbers instead of decimals
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum**-1 # we do this instead of dividing by counts cos easier to calculus
probs = counts * counts_sum_inv # divide the raw count by the summed count to get probabilities
logprobs = probs.log() # take log of probabilities
loss = -logprobs[range(n), Yb].mean() # for each obs in the batch, get the log likelihood corresponding to the correct Y value, take the negative and then average them

# backward pass, replacing loss.backward()
for p in params:
    p.grad = None

for t in [logprobs, probs, counts_sum_inv, counts_sum, counts, norm_logits, logit_maxes, logits, h, hpreact, bnraw, bnvar_inv, bnvar, bndiff2, bndiff, bnmeani, hprebn, embcat, emb]:
    t.retain_grad()
loss.backward()
loss

tensor(3.3647, grad_fn=<NegBackward0>)

In [9]:
bndiff.shape

torch.Size([32, 64])

In [ ]:
# Exercise 1: backprop through the whole thing manually, 
# backpropagating through exactly all of the variables 
# as they are defined in the forward pass above, one by one

# -----------------
# note - the derivative term will obviously always be the same shape as the original
# another note: a summation in the forward pass will likely mean a replication/broadcasting in the backward pass, and vice versa

# start with dlogprobs
# we want dloss/dlogprobs
dlogprobs = torch.zeros_like(logprobs) # start with zeros - exact same shape as logprobs. We start with zeros since by default, 26 out of 27 of the logprobs (all but the one corresponding to the Y value) will not impact loss
dlogprobs[range(n), Yb] = -1.0/n # for the logprobs that actually get chosen (because they directly impact loss, as they are the ones that correspond to the Y values), we set them to -1/batchsize due to the average
# for example if we have y = -(a+b+c)/3, dy/da = -1/3, as for dy/db and dy/dc, so we can extend that to -1/n


# now we want dloss/dprobs
# naturally breakup with chain rule to get dloss/dprobs = dloss/dlogprobs * dlogprobs/dprobs
# we have dloss/dlogprobs from above. we also have logprobs = ln(probs), so dlogprobs/dprobs = 1/probs
dprobs = dlogprobs * 1.0/probs
# intuitively, we are boosting the gradient of the observations with very low probability, which will flow through the correct nodes due to the sparse nature of dlogprobs


# next step - dloss/dcounts_sum_inv
# break it up into dloss/dprobs * dprobs/dcounts_sum_inv
# we have probs = counts * counts_sum_inv, so dprobs/dcounts_sum_inv = counts
# orinarily this would just be dprobs * counts from chain rule
# BUT counts_sum_inv has shape 32, 1 so there is an implicit broadcasting
# simple example: c = a * b where a[3x3], b[3,1]
# a11*b1 a12*b1 a13*b1
# a21*b2 a22*b2 a23*b2
# a31*b3 a32*b3 a33*b3 # notice the b1, b2, b3 get broadcast horizontally to give c[3,3]
# split this into two operations: the multiplication step and the broadcast step
# dprobs * counts is the derivative from just the multiplication step i.e. the derivative after weve already done the broadcasting
# recall from micrograd, if a node splits out into multiple branches, its gradient gets added together across all of its child nodes
# for example, since the b1 is used across all of the columns, we then need to sum the entire matrix across all of the columns to get the final total gradient for the b1 - hence the sum(1), and then keepdim so dcount_sum_inv is the same same as count_sum_inv
dcounts_sum_inv = (dprobs * counts).sum(1, keepdim=True)


# next step - dloss/dcounts_sum
# counts_sum is also a column vector so keep that in mind
# dloss/dcounts_sum broken into dloss/dcounts_sum_inv * dcounts_sum_inv/dcounts_sum
# dcounts_sum_inv, counts_sum are both column vectors, so we can simply do element wise
# counts_sum_inv = 1/counts_sum, so dcounts_sum_inv/dcounts_sum = -1/counts_sum^2
dcounts_sum = dcounts_sum_inv * -counts_sum**-2


# next step - dloss/dcounts
# now we have to be careful cos counts is 32x27, and our preceding steps will be column vectors
# also have to be careful because counts actually gets used twice: 1. goes into counts_sum 2. goes straight into probs
# split the chain rule TWICE, for the two branches that counts takes on its way to loss
# dloss/dcounts =       (dloss/dcounts_sum * dcounts_sum/dcounts)     +      (dloss/dprobs * dprobs/dcounts)
# 1. dloss/dcounts_sum * dcounts_sum/dcounts: since it is a summation node, the gradient from dloss/dcounts_sum just passes right thru, but we need to broadcast it to the correct shape horizontally
dcounts1 = torch.ones_like(counts) * dcounts_sum
# 2. dloss/dprobs * dprobs/dcounts: since probs = counts * counts_sum_inv, we get dprobs/dcounts = counts_sum_inv
dcounts2 = dprobs * counts_sum_inv
# add the two together
dcounts = dcounts1 + dcounts2


# next step - dloss/dnorm_logits
# chain rule, break into dloss/dnorm_logits = dloss/dcounts * dcounts/dnorm_logits
# we have counts = e^norm_logits, so simple derivative is same function dcounts/dnorm_logits = e^norm_logits
dnorm_logits = dcounts * norm_logits.exp()

# next step - dloss/dlogit_maxes
# chain rule, break into dloss/dlogit_maxes = dloss/dnorm_logits * dnorm_logits/dlogit_maxes
# need to be careful since logit maxes is a column vector again
# norm_logits = logits - logit_maxes so dnorm_logits/dlogit_maxes = -1
# however as with the counts_sum_inv, the column vector of logit maximums gets broadcast into the matrix of logits
# so after the chain rule, we sum across the columns to aggregate the gradient of all the places where each logit max was applied/replicated/broadcast
dlogit_maxes = (-1.0 * dnorm_logits).sum(1, keepdim=True)

# next step dloss/dlogits
# note that logits gets used in logit_maxes and norm logits
# chain rule, break into dloss/dlogits =    dloss/dnorm_logits*dnorm_logits/dlogits    +    dloss/dlogit_maxes*dlogit_maxes/dlogits
# 1. dloss/dnorm_logits*dnorm_logits/dlogits: norm_logits = logits - logit_maxes so dnorm_logits/dlogits = 1
dlogits1 = dnorm_logits.clone() # multiplying by 1 obvs creates same vector, use clone() for safety
# 2. dloss/dlogit_maxes*dlogit_maxes/dlogits: logit_maxes = max(logits), so the derivative is 1 for the maximum value, 0 for all othersm
dlogits2 = F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes
# this is a quicker implementation instead of initialising all 0s and setting specific indices to 1 - use one hot encoding instead for the specific indices
# also note that dlogit_maxes is a column vector that is getting broadcast, and due to the one hot, it only gets routed through the indices that were chosen as maximum
dlogits = dlogits1 + dlogits2

# next step dloss/dh
# backprop thru matrix multiplication
# note the shapes: h[32, 64], W2[64, 27], b[1, 27], logits[32, 27]. formula logits = h @ W2 + b2
# referring to the notebook matmul, if we have d = a@b + c, we get: 
# dL/da = dL/dd @ bT
# dL/db = aT @ dL/dd
# dL/dc = dL/dd.sum(0) sum across the columns to give a row vector (can think of it as eliminating the 0th dimension, going from shape [32, 27] to [27])
# we apply this formula: dL/da = dL/dd @ bT where d=logits, a=h, b=weights
# note: you dont even need to remember the formula, just consider what combination of logits and weights will give you the shape required to match h
dh = dlogits @ W2.T

# next step dloss/dW2
# using same formulas as above: d = a@b + c gives dL/db = aT @ dL/dd, but we have d=logits, a=h, b=weights
dW2 = h.T @ dlogits

# next step dloss/db2
# same formula for dL/dc from above, no need to keepdim
db2 = dlogits.sum(0) 

# linear layer done, now we want dloss/dhpreact
# chain rule: dloss/dhpreact = dloss/dh * dh/dhpreact
# given h = tanh(hpreact), we can get dh/dhpreact = 1 - tanh(hpreact)^2 = 1 - h^2
# dhpreact = (1.0 - h**2) * dh
dhpreact = hpreact.grad # we just use the torch gradient for this since the math isnt exact, and the floating point error flows on all the way thru, but the calc above is correct

# going thru the batchnorm layer now - we want dloss/dbngain
# quick note on shapes: gain and bias have shape [1, 64] i.e. they are row vectors so they get broadcast across each row, as in they get applied to every obs in the data
# we also have bnraw[32, 64]
# chain rule dloss/dbngain = dloss/dhpreact * dhpreact/dbngain
# with hpreact = bngain * bnraw + bnbias, we get dhpreact/dbngain = bnraw
# we then have to derive the broadcasting step, by summing up all of the places where bngain was applied, i.e. across each row
dbngain = (bnraw * dhpreact).sum(0, keepdim=True)

# now for dloss/dbnbias
# same thing as above, but the local derivative coming through the chain rule is 1
# recall the addition rule from micrograd, the gradients just flow through when we add, but we just need to handle the broadcasting by summing up all of their appearances
dbnbias = dhpreact.sum(0, keepdim=True)

# finally for dloss/dbnraw
# chain: dloss/dbnraw = dloss/dhpreact * dhpreact/dbnraw
# with hpreact = bngain * bnraw + bnbias, we get dhpreact/dbnraw = dbngain
# note this operation will broadcast the gain across each column
dbnraw = dhpreact * bngain

# now we get dloss/dbnvar_inv
# chain rule dloss/dbnvar_inv = dloss/dbnraw * dbnraw/dbnvar_inv
# we have bnraw = bndiff * bnvar_inv, so dbnraw/dbnvar_inv = bndiff
# check shapes though - similar to gain and bias, the var is [1, 64], i.e. a row vector that gets applied to each datapoint
# but bndiff and dbnraw are [32, 64], so sum across the rows to get a row vector
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)

# now for dloss/dbnvar
# chain rule dloss/dbnvar = dloss/dbnvar_inv * dbnvar_inv/dbnvar
# we have bnvar_inv = (bnvar + 1e-5)**-0.5, so dbnvar_inv/dbnvar = -0.5 * (bnvar + 1e-5) ** -1.5
# both are row vectors so no worries
dbnvar = dbnvar_inv * (-0.5*(bnvar + 1e-5)**-1.5)

# now for dloss/dbndiff2
# shape dbndiff2[32, 64], but bnvar is [1, 64]
# chain rule dloss/dbndiff2 = dloss/dbnvar * dbnvar/dbndiff2
# formula is bnvar = 1/(n-1) * bndiff2.sum(0, keepdim=True) -> so the local derivative of the sums just becomes 1/n-1
# since its a sum we need to broadcast it to the [32, 64] shape which we can do by setting up ones_like and then the multiplication will automatically handle it
dbndiff2 = torch.ones_like(bndiff2) * 1.0/(n-1) * dbnvar

# now for dloss/dbndiff
# appears in two places, so our chain rule becomes dloss/dbndiff =  dloss/dbndiff2 * dbndiff2/dbndiff   +   dloss/dbnraw * dbnraw/dbndiff
# all of the steps are shape [32, 64] so no worries
# 1. dloss/dbndiff2 * dbndiff2/dbndiff: we have bndiff2 = bndiff**2, so  dbndiff2/dbndiff = 2bndiff
dbndiff_1 = (2*bndiff) * dbndiff2
# 2. dloss/dbnraw * dbnraw/dbndiff: we have bnraw = bndiff * bnvar_inv so dbnraw/dbndiff = bnvar_inv
dbndiff_2 = bnvar_inv * dbnraw
dbndiff = dbndiff_1 + dbndiff_2

# now for dloss/dbnmeani
# chain rule: dloss/dbnmeani = dloss/dbndiff * dbndiff/dbnmeani
# note bnmeani is a row vector [1, 64], so we need some sort of summation here to go from bndiff, which was broadcasted in the forward pass  
# we have bndiff = hprebn - bnmeani so dbndiff/bnmeani = -1
dbnmeani = (-dbndiff).sum(0)


# dloss/dhprebn
# used in 2 places chain rule: dloss/dhprebn = dloss/dbndiff*dbndiff/dhprebn    +   dloss/dbnmeani*dbmeani/dhprebn
# shapes: hprebn and diff both [32, 64] so all good, but meani needs to be broadcast
# 1. dloss/dbndiff*dbndiff/dhprebn:     bndiff = hprebn - bnmeani so dbndiff/dhprebn = 1
dhprebn1 = dbndiff.clone()
# 2. dloss/dbnmeani*dbmeani/dhprebn: we have bnmeani = 1/n * hprebn.sum(0, keepdim=True) -> dbmeani/dhprebn gives 1/n but needs to be broadcast to the same dim
dhprebn2 = torch.ones_like(hprebn) * 1/n * dbnmeani
dhprebn = dhprebn1 + dhprebn2


# dloss/embcat - now we do linear layer
# we have hprebn = embcat @ W1 + b1
# use our formula from previous linear layer
# shapes are embcat[32, 30], hprebn[32, 64], W1[30, 64], b1[64]
# just going off the shapes, in order to use our hprebn and W1 to get the same shape as embcat, we need to do hprebn[32,64] @W1.T[64, 30] which will give [32, 30]
dembcat = dhprebn @ W1.T

# dloss/dW1
# same implementation as the formula above - to get W1 using emb and hprebn, we need emb.T[30, 32] @ hprebn[32, 64] which gives [30,64]
dW1 = embcat.T @ dhprebn

# dloss/db1
# given our hprebn[32, 64], to get to the [64] required, we just sum across the 0th dimension and no need to keepdim since it is a directionless tensor
db1 = dhprebn.sum(0)

# dloss/demb
# chain rule dloss/demb = dloss/dembcat * dembcat/demb
# dembcat has shape [32, 30], emb has shape[32, 3, 10], note there is no transformation here, just shaping so the gradients just flow through but in a different shape
demb = dembcat.view(emb.shape)

# finally, dloss/dC
# chain rule dloss/demb * demb/dC
# emb has shape[32, 3, 10], C has shape [27, 10], Xb has shape [32, 3]
# once again, there were no transformations so the gradient would just flow through, but we just need to reverse the indexing from emb = C[Xb]
# go through X one value at a time, get the index value, go to that corresponding index in C which represents the respective character, and add the demb which corresponds to that character embedding
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        dC[Xb[k, j]] += demb[k, j]


# -----------------

cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
cmp('C', dC, C)

logprobs        | exact: True  | approximate True  | maxdiff: 0.0
probs           | exact: True  | approximate True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate True  | maxdiff: 0.0
counts          | exact: True  | approximate True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate True  | maxdiff: 0.0
logits          | exact: True  | approximate True  | maxdiff: 0.0
h               | exact: True  | approximate True  | maxdiff: 0.0
W2              | exact: True  | approximate True  | maxdiff: 0.0
b2              | exact: True  | approximate True  | maxdiff: 0.0
hpreact         | exact: True  | approximate True  | maxdiff: 0.0
bngain          | exact: True  | approximate True  | maxdiff: 0.0
bnbias          | exact: True  | approximate True  | maxdiff: 0.0
bnraw           | exact: True  | approximate True  | maxdiff: 0.0
bnvar_inv 

In [11]:
# Exercise 2: backprop through cross_entropy but all in one go
# to complete this challenge look at the mathematical expression of the loss,
# take the derivative, simplify the expression, and just write it out

# forward pass

# before:
# logit_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logit_maxes # subtract max for numerical stability
# counts = norm_logits.exp()
# counts_sum = counts.sum(1, keepdims=True)
# counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
# probs = counts * counts_sum_inv
# logprobs = probs.log()
# loss = -logprobs[range(n), Yb].mean()

# now:
loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())

3.3646762371063232 diff: 0.0


In [12]:
logits.shape

torch.Size([32, 27])

In [13]:
# backward pass

# dont need chain rule here, we are just calculating dloss/dlogits direct from formula
# note dlogits is shape[32, 27] when loss is just a single number

# refer notebook, in the cases that dont correspond to the y index, dloss/dlogit is just equal to the softmax
# in the cases that line up with the y index, dloss/dlogit is softmax-1

dlogits = F.softmax(logits, 1) # take the softmax for all logits
dlogits[range(n), Yb] -= 1 # subtract 1 from the ones which correspond to the y index
dlogits /= n # scale by n since we take the average

cmp('logits', dlogits, logits) # I can only get approximate to be true, my maxdiff is 6e-9

logits          | exact: False | approximate True  | maxdiff: 7.2177499532699585e-09


In [14]:
# Interesting note
# consider the logits for a row - notice that the sum is actually 0, with one large negative in there. This one comes from the one corresponding to the correct y
# and is the one that we subtracted one from. The force pulling up the probability of the correct index is actually equal to the force pushing down the probability of all of the incorrect indexes
dlogits[0]
# dlogits[0].sum()

tensor([ 0.0021,  0.0027,  0.0006,  0.0016,  0.0006,  0.0025,  0.0009,  0.0012,
        -0.0306,  0.0010,  0.0012,  0.0012,  0.0012,  0.0009,  0.0011,  0.0004,
         0.0003,  0.0006,  0.0005,  0.0017,  0.0015,  0.0007,  0.0008,  0.0022,
         0.0018,  0.0008,  0.0007], grad_fn=<SelectBackward0>)

In [15]:
# Exercise 3: backprop through batchnorm but all in one go
# to complete this challenge look at the mathematical expression of the output of batchnorm,
# take the derivative w.r.t. its input, simplify the expression, and just write it out
# BatchNorm paper: https://arxiv.org/abs/1502.03167

# forward pass

# before:
# bnmeani = 1/n*hprebn.sum(0, keepdim=True)
# bndiff = hprebn - bnmeani
# bndiff2 = bndiff**2
# bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
# bnvar_inv = (bnvar + 1e-5)**-0.5
# bnraw = bndiff * bnvar_inv
# hpreact = bngain * bnraw + bnbias

# now:
hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5) + bnbias
print('max diff:', (hpreact_fast - hpreact).abs().max())

max diff: tensor(4.7684e-07, grad_fn=<MaxBackward1>)


In [16]:
hpreact.shape, hprebn.shape

(torch.Size([32, 64]), torch.Size([32, 64]))

In [21]:
# backward pass

# before we had:
# dbnraw = bngain * dhpreact
# dbndiff = bnvar_inv * dbnraw
# dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
# dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv
# dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar
# dbndiff += (2*bndiff) * dbndiff2
# dhprebn = dbndiff.clone()
# dbnmeani = (-dbndiff).sum(0)
# dhprebn += 1.0/n * (torch.ones_like(hprebn) * dbnmeani)

# calculate dhprebn given dhpreact (i.e. backprop through the batchnorm)
# (you'll also need to use some of the variables from the forward pass up above)

# chain rule dloss/dhprebn = dloss*/dhpreact * dhpreact/dhprebn
# note shapes are both 32,64

# -----------------
# YOUR CODE HERE :)
dhprebn = bngain*bnvar_inv/n * ((n*dhpreact) - dhpreact.sum(0) - (n/(n-1)) * bnraw * (dhpreact * bnraw).sum(0))
# -----------------

cmp('hprebn', dhprebn, hprebn) # I can only get approximate to be true, my maxdiff is 9e-10

hprebn          | exact: False | approximate True  | maxdiff: 7.450580596923828e-09
